In [ ]:
# Cell 1: Mount Google Drive (holds the real datasets at
# /content/drive/MyDrive/plantguard-data/, per src/config.py).
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Cell 2: Clone or pull the private repo using a PAT entered via getpass
# (the token is never hardcoded or printed/logged), then purge any
# already-imported src.* modules so a stale cached module from an earlier
# cell run in this session can never silently run instead of the code just
# pulled, and print the commit actually checked out.
import os
import subprocess
import sys
from getpass import getpass

REPO_DIR = "/content/plantguard-v2"
pat = getpass("GitHub Personal Access Token: ")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    remote = f"https://{pat}@github.com/RUDRAIndia/plantguard-v2.git"
    subprocess.run(["git", "clone", remote, REPO_DIR], check=True)

del pat
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

purged = sorted(name for name in sys.modules if name.startswith("src"))
for name in purged:
    del sys.modules[name]
print(f"Purged {len(purged)} cached src module(s): {purged}")

commit_hash = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print(f"Checked out commit {commit_hash}")

In [ ]:
# Cell 3: Kaggle credentials. See src/data/kaggle_auth.py for the accepted
# kaggle.json shapes, where credentials are installed, and the
# authentication proof that runs before any download starts.
from src.data import kaggle_auth

kaggle_auth.install_credentials()

In [ ]:
# Cell 4: Download PlantVillage and PlantDoc.
from src.data import download

download.main()

In [ ]:
# Cell 5: Build the dataset inventory and the PlantDoc-to-PlantVillage
# class mapping.
from src.data import inventory, mapping_report

inventory.main()
mapping_report.main()

In [ ]:
# Cell 6: Print the inventory report.
from src import config

print((config.ARTIFACTS_DIR / "inventory.md").read_text(encoding="utf-8"))

In [ ]:
# Cell 7: Deduplicate PlantVillage (group near-duplicate leaves so none
# span train/val/test) and build the leakage-free, stratified split.
from src.data import dedupe, split_report

dedupe.main()
split_report.main()

In [ ]:
# Cell 8: Print the dedupe and split reports.
from src import config

print((config.ARTIFACTS_DIR / "dedupe_report.md").read_text(encoding="utf-8"))
print((config.ARTIFACTS_DIR / "split_report.md").read_text(encoding="utf-8"))

In [ ]:
# Cell 9: Build tf.data pipelines, run augmentation sanity checks, print
# class weights and batch shapes.
from src import config
from src.data import pipeline, sanity

train_ds, val_ds, test_ds, class_weights = pipeline.build_datasets(
    model_name=config.CANDIDATE_MODELS[0]  # placeholder; real choice is made later, on val only
)
sanity.main()

print("Class weights:", class_weights)
for name, ds in (("train", train_ds), ("val", val_ds), ("test", test_ds)):
    images, labels = next(iter(ds.take(1)))
    print(f"{name}: images {images.shape}, labels {labels.shape}")


In [ ]:
# Cell 10: Download the not-a-leaf negative set and report what was
# assembled and where it was persisted on Drive.
from src import config
from src.data import negatives, validate

negatives_dir = negatives.download_negatives()
image_count = validate.count_images(negatives_dir)

print(f"Negative images assembled: {image_count} at {negatives_dir}")
print(f"Persisted to Drive tar at {config.NEGATIVES_TAR}")


In [ ]:
# Cell 11: Run the training smoke test for one model end to end (frozen
# head phase + fine-tune phase, on a small synthetic image set), then print
# the resulting history JSON -- verifies the whole training path before
# committing to a real run.
import json

from src import config, train

manifest = train.run_training(model_name=config.CANDIDATE_MODELS[0], smoke=True)
print(json.dumps(manifest, indent=2))

history_path = config.ARTIFACTS_DIR / f"history_{config.CANDIDATE_MODELS[0]}.json"
print(history_path.read_text(encoding="utf-8"))
